## Task 1


Download Dataset For Finetuning

In [ ]:
!pip install roboflow

from roboflow import Roboflow
rf = Roboflow(api_key="ip0p7dPIY908ZIFltm0g")
project = rf.workspace("dronenotdrone").project("drone-detection-pq8sj")
version = project.version(2)
dataset = version.download("yolo26")
                

Finetune Model

In [ ]:
from ultralytics import YOLO

model = YOLO('yolo26n.pt')
# this dataset actually contains birds as well which i thought would be good to prevent accidental detections (0=drone, 1=bird)
model.train(data='drone-detection-2/data.yaml', epochs=20, warmup_epochs=0, imgsz=640, plots=True, device=1)

Extract all video frames

In [24]:
import glob
import os

def extract_frames(video_dir="videos", frames_dir="frames", fps=5):
    import subprocess
    video_paths = glob.glob(f"{video_dir}/*.mp4")
    
    if not video_paths:
        print(f"No .mp4 files found in '{video_dir}/'")
        return

    for video_path in video_paths:
        video_name = os.path.splitext(os.path.basename(video_path))[0]
        out_folder = os.path.join(frames_dir, video_name)
        os.makedirs(out_folder, exist_ok=True)

        subprocess.run([
            "ffmpeg", "-y", "-i", video_path,
            "-vf", f"fps={fps}",
            os.path.join(out_folder, "frame_%04d.jpg"),
            "-loglevel", "error"
        ], check=True)

        print(f"Extracted → {out_folder}")

Run detections on all frames in a folder

In [25]:
import glob
import os
import pandas as pd
import cv2

from tqdm import tqdm
from ultralytics import YOLO

In [27]:
def detect(model, frames_folder="frames", batch_size=16, confidence=0.331):
    image_names = []
    video_ids = []
    detected_centers = []
    confidence_scores = []

    frame_paths = sorted(glob.glob(frames_folder + "/**/*.jpg", recursive=True))

    for i in tqdm(range(0, len(frame_paths), batch_size), desc="Processing Frames"):
        batch = frame_paths[i : i+batch_size]
        results = model.predict(batch, conf=confidence, classes=[0])

        for j in range(len(results)):
            result = results[j]
            num_preds = len(result.boxes)

            if num_preds > 0:
                img = result.plot()
                center_points = []
                for box in result.boxes:
                    coords = box.xyxy[0]
                    cx = ((coords[0] + coords[2]) / 2).item()
                    cy = ((coords[1] + coords[3]) / 2).item()
                    center_points.append((cx, cy))

                filename = os.path.basename(batch[j])
                video_id = os.path.basename(os.path.dirname(batch[j]))
                
                os.makedirs(f"detections/{video_id}", exist_ok=True)
                cv2.imwrite(f"detections/{video_id}/{filename}", img)



                image_names.append(filename)
                video_ids.append(video_id)
                detected_centers.append(center_points)
                confidence_scores.append(result.boxes.conf.tolist())

                
    data = {
        "image_name": image_names,
        "video_ids": video_ids,
        "bounding_box_center": detected_centers,
        "confidence_score": confidence_scores
    }

    df = pd.DataFrame(data)

    df.to_parquet('detections/drone_detections.parquet', engine='pyarrow', compression='snappy')

    print("File saved successfully!")

## Task 2

In [28]:
from filterpy.kalman import KalmanFilter
import numpy as np

Initialize Kalman FIlter Function

In [29]:
def make_kalman(cx, cy, meas_noise=5.0, proc_noise=0.1):
    kf = KalmanFilter(dim_x=4, dim_z=2)
 
    # State transition — constant velocity model
    kf.F = np.array([
        [1, 0, 1, 0],
        [0, 1, 0, 1],
        [0, 0, 1, 0],
        [0, 0, 0, 1],
    ], dtype=float)
 
    # Measurement matrix — we observe cx, cy only
    kf.H = np.array([
        [1, 0, 0, 0],
        [0, 1, 0, 0],
    ], dtype=float)
 
    kf.R *= meas_noise       # measurement noise
    kf.Q *= proc_noise    # process noise
 
    kf.x = np.array([[cx], [cy], [0.], [0.]])
    kf.P *= 10               # initial uncertainty
 
    return kf

In [30]:
def run_tracking(video_name, frame_paths, parquet_path, output_dir, meas_noise, proc_noise, max_miss, fps):

    df = pd.read_parquet(parquet_path)
    detections = {
        (row["video_ids"], row["image_name"]): row["bounding_box_center"]
        for _, row in df.iterrows()
    }

    os.makedirs(output_dir, exist_ok=True)

    sample = cv2.imread(frame_paths[0])
    h, w   = sample.shape[:2]

    out_path = os.path.join(output_dir, f"{video_name}_tracked.mp4")
    fourcc   = cv2.VideoWriter_fourcc(*"mp4v")
    writer   = cv2.VideoWriter(out_path, fourcc, fps, (w, h))

    kf         = None
    miss_count = 0
    trajectory = []

    for frame_path in tqdm(frame_paths, desc=f"Tracking [{video_name}]"):
        filename = os.path.basename(frame_path)
        key      = (video_name, filename)
        detected = key in detections

        if detected:
            centers = detections[key]
            cx, cy  = centers[0]

            # Load pre-annotated frame from detections folder
            det_img_path = os.path.join("detections", video_name, filename)
            img = cv2.imread(det_img_path)

            if kf is None:
                kf = make_kalman(cx, cy, meas_noise, proc_noise)
            else:
                kf.predict()
                kf.update(np.array([[cx], [cy]]))

            miss_count = 0
            est_cx = float(kf.x[0][0])
            est_cy = float(kf.x[1][0])
            trajectory.append((int(est_cx), int(est_cy)))

            # No drawing needed — bbox already in the image from result.plot()

        elif kf is not None:
            # Coast — load raw frame from frames folder
            img = cv2.imread(frame_path)
            kf.predict()
            miss_count += 1

            if miss_count > max_miss:
                kf         = None
                trajectory = []
                continue

            est_cx = float(kf.x[0][0])
            est_cy = float(kf.x[1][0])
            trajectory.append((int(est_cx), int(est_cy)))

            cv2.circle(img, (int(est_cx), int(est_cy)), 4,
                       (0, 165, 255), -1)

        else:
            continue

        # Draw trajectory polyline on whichever img was loaded
        if len(trajectory) >= 2:
            pts = np.array(trajectory, dtype=np.int32).reshape(-1, 1, 2)
            cv2.polylines(img, [pts], isClosed=False,
                          color=(0, 0, 255), thickness=2)

        writer.write(img)

    writer.release()
    print(f"Output video → {out_path}")

## Put them Together

In [31]:
model_path = "models/finetuned_100epochs.pt"
model = YOLO(model_path)

VIDEO_DIR = "videos"
FRAME_OUTPUT_DIR = "frames"
OUTPUT_DIR = "outputs"

MEASURE_NOISE = 5.0
PROCESS_NOISE   = 0.1

MAX_MISS = 5
FPS = 5

In [32]:
extract_frames(video_dir=VIDEO_DIR, frames_dir=FRAME_OUTPUT_DIR, fps=FPS)

Extracted → frames/drone_video_1
Extracted → frames/drone_video_2


In [33]:
os.makedirs("detections", exist_ok=True)

detect(model, frames_folder="frames", batch_size=16, confidence=0.331)

0: 384x640 1 0, 115.4ms
1: 384x640 1 0, 115.4ms
2: 384x640 1 0, 115.4ms
3: 384x640 1 0, 115.4ms
4: 384x640 1 0, 115.4ms
5: 384x640 1 0, 115.4ms
6: 384x640 1 0, 115.4ms
7: 384x640 1 0, 115.4ms
8: 384x640 1 0, 115.4ms
9: 384x640 1 0, 115.4ms
10: 384x640 1 0, 115.4ms
11: 384x640 1 0, 115.4ms
12: 384x640 1 0, 115.4ms
13: 384x640 1 0, 115.4ms
14: 384x640 1 0, 115.4ms
15: 384x640 1 0, 115.4ms
Speed: 1.5ms preprocess, 115.4ms inference, 0.1ms postprocess per image at shape (1, 3, 384, 640)


Processing Frames:   3%|▎         | 7/213 [00:16<07:56,  2.31s/it]


0: 384x640 1 0, 130.3ms
1: 384x640 1 0, 130.3ms
2: 384x640 1 0, 130.3ms
3: 384x640 1 0, 130.3ms
4: 384x640 1 0, 130.3ms
5: 384x640 1 0, 130.3ms
6: 384x640 1 0, 130.3ms
7: 384x640 1 0, 130.3ms
8: 384x640 1 0, 130.3ms
9: 384x640 1 0, 130.3ms
10: 384x640 1 0, 130.3ms
11: 384x640 1 0, 130.3ms
12: 384x640 1 0, 130.3ms
13: 384x640 1 0, 130.3ms
14: 384x640 1 0, 130.3ms
15: 384x640 1 0, 130.3ms
Speed: 1.5ms preprocess, 130.3ms inference, 0.1ms postprocess per image at shape (1, 3, 384, 640)


Processing Frames:   4%|▍         | 8/213 [00:19<08:04,  2.36s/it]


0: 384x640 1 0, 129.3ms
1: 384x640 1 0, 129.3ms
2: 384x640 1 0, 129.3ms
3: 384x640 1 0, 129.3ms
4: 384x640 1 0, 129.3ms
5: 384x640 1 0, 129.3ms
6: 384x640 1 0, 129.3ms
7: 384x640 1 0, 129.3ms
8: 384x640 1 0, 129.3ms
9: 384x640 1 0, 129.3ms
10: 384x640 1 0, 129.3ms
11: 384x640 1 0, 129.3ms
12: 384x640 1 0, 129.3ms
13: 384x640 1 0, 129.3ms
14: 384x640 1 0, 129.3ms
15: 384x640 1 0, 129.3ms
Speed: 1.8ms preprocess, 129.3ms inference, 0.1ms postprocess per image at shape (1, 3, 384, 640)


...


0: 384x640 (no detections), 119.0ms
1: 384x640 (no detections), 119.0ms
2: 384x640 (no detections), 119.0ms
3: 384x640 (no detections), 119.0ms
4: 384x640 (no detections), 119.0ms
5: 384x640 (no detections), 119.0ms
6: 384x640 (no detections), 119.0ms
7: 384x640 (no detections), 119.0ms
8: 384x640 (no detections), 119.0ms
9: 384x640 (no detections), 119.0ms
10: 384x640 (no detections), 119.0ms
11: 384x640 (no detections), 119.0ms
12: 384x640 (no detections), 119.0ms
13: 384x640 (no detections), 119.0ms
14: 384x640 (no detections), 119.0ms
15: 384x640 (no detections), 119.0ms
Speed: 1.7ms preprocess, 119.0ms inference, 0.1ms postprocess per image at shape (1, 3, 384, 640)


Processing Frames: 100%|█████████▉| 212/213 [07:36<00:02,  2.18s/it]


0: 384x640 (no detections), 113.6ms
1: 384x640 (no detections), 113.6ms
2: 384x640 (no detections), 113.6ms
3: 384x640 (no detections), 113.6ms
4: 384x640 (no detections), 113.6ms
5: 384x640 (no detections), 113.6ms
6: 384x640 (no detections), 113.6ms
7: 384x640 (no detections), 113.6ms
8: 384x640 (no detections), 113.6ms
9: 384x640 (no detections), 113.6ms
10: 384x640 (no detections), 113.6ms
11: 384x640 (no detections), 113.6ms
12: 384x640 (no detections), 113.6ms
13: 384x640 (no detections), 113.6ms
14: 384x640 (no detections), 113.6ms
15: 384x640 (no detections), 113.6ms
Speed: 1.6ms preprocess, 113.6ms inference, 0.1ms postprocess per image at shape (1, 3, 384, 640)


Processing Frames: 100%|██████████| 213/213 [07:38<00:00,  2.15s/it]

File saved successfully!


In [34]:
video_names = [
    d for d in os.listdir("frames")
    if os.path.isdir(os.path.join("frames", d))
]

for video_name in video_names:
    frame_paths = sorted(glob.glob(f"frames/{video_name}/*.jpg"))

    run_tracking(
        video_name   = video_name,
        frame_paths  = frame_paths,
        parquet_path = "detections/drone_detections.parquet",
        output_dir   = OUTPUT_DIR,
        meas_noise   = MEASURE_NOISE,
        proc_noise   = PROCESS_NOISE,
        max_miss     = MAX_MISS,
        fps          = FPS,
    )

Tracking [drone_video_1]: 100%|██████████| 828/828 [00:06<00:00, 120.39it/s]


Output video → outputs/drone_video_1_tracked.mp4


Tracking [drone_video_2]: 100%|██████████| 2580/2580 [00:01<00:00, 1394.16it/s]

Output video → outputs/drone_video_2_tracked.mp4
